# Loss sensitivities for one unseen target cell: with and without the latent skip

Does removing the latent skip widen the model's radius of influence? This notebook compares
checkpoints on the same date and target cell, for the `cell` scalar: `ObservationMaskedMSE` on
sla at one target cell that no observed input bin saw. The globe-wide loss is in
`sensitivity_global.ipynb`.

Both checkpoints have the same task and inputs: static forcings enter once, at t0, and the
time-varying forcings also at the target time, as a fourth input step (input hour +24) that
carries no observations. They differ only in `latent_skip`:
- `skip` is `../update-forcings-aa/runs/v0`, with the latent skip
- `no_skip` is this experiment's `runs/v0`, without it

The gradients are computed by `sensitivity.py` on a GPU, in `nese-anemoi` (`sbatch submit_sensitivity.sh`).
Each case in that script is one date and one target cell, written to
`sensitivity/<case>/<label>.zarr`. This notebook reads one case, set by `CASE` (or `SENSITIVITY_CASE`),
and runs in `nese`. Shared loading and drawing is in `sensitivity_plots.py`. Cases and their
target cells follow `../unseen-ft-all-altimeter`.

The gradients:
- **latent** is `|dL/dz|`, the norm over channels at each hidden (hpx6) node, where `z` is the processor input.
- **decoder** is `|g|`, the gradient at the decoder's hidden-node input. With `latent_skip` (`skip`) the decoder sees `processor(z) + z`, so `g` is the skip path alone and latent is `J_procᵀ g + g`. Without it (`no_skip`) the decoder sees `processor(z)`, so `g` is the gradient at the processor's output.
- **processor** is `|J_procᵀ g|`, what reached `z` through the processor: `latent − g` for `skip`, and latent itself for `no_skip`.
- **input** is `|dL/dx|`, summed over the variables of each kind (all sla missions, ...). `x` is the normalised, imputed input at each input hour.
- **encoder input** is the same at the encoder's input: each field's FieldEncoder latent (8 channels, at every cell, observed or not) and the passthrough inputs.

Maps are `log10` of the sensitivity over its maximum, clipped at `LOG_FLOOR`, so the checkpoints are compared by shape, not magnitude.

For the `cell` scalar, the decoder gradient is nonzero only at the target's 3 decoder
neighbours, so **processor** is the one to read for the processor's radius of influence.

In [ ]:
import os

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sensitivity_plots as plots
import xarray as xr
from cartopy.geodesic import Geodesic
from sensitivity_plots import load
from sensitivity_plots import map_axes
from sensitivity_plots import normalised_log
from sensitivity_plots import plot_shares

# Which run of sensitivity.py to read; submit_sensitivity_notebook.sh sets it per case
CASE = os.environ.get("SENSITIVITY_CASE", "gulf_stream")
CHECKPOINTS = ["skip", "no_skip"]
EARTH_RADIUS_KM = 6371.0
DISTANCE_BINS_KM = np.arange(0, 6001, 100)
ZOOM_DEGREES = 25
# Graph radii, from training.yaml: encoder CutOffEdges (data -> hidden) and processor stencil
RADII_KM = {"encoder, 100 km": 100, "processor stencil, 1200 km": 1200}
RADII_STYLES = ["--", ":"]  # same order as RADII_KM, in the line plots and on the maps

In [ ]:
sens = load(CASE, CHECKPOINTS)
cell = sens.sel(scalar="cell")
target_lat, target_lon = sens.attrs["target_latitude"], sens.attrs["target_longitude"]
ZOOM = [target_lon - ZOOM_DEGREES, target_lon + ZOOM_DEGREES, target_lat - ZOOM_DEGREES, target_lat + ZOOM_DEGREES]
print(f"case {sens.attrs['case']}, requested point {sens.attrs['requested_point']}")
print(f"t0 {sens.attrs['t0']} -> target {sens.attrs['target_time']}")
print(f"target cell {sens.attrs['target_cell']} at ({target_lat:.2f}, {target_lon:.2f})")
cell[["loss", "loss_scale"]].to_dataframe()

## Distance from the target cell

In [ ]:
def great_circle_km(lat: xr.DataArray, lon: xr.DataArray) -> xr.DataArray:
    lat, lon, lat0, lon0 = (np.radians(v) for v in (lat, lon, target_lat, target_lon))
    cosine = np.sin(lat) * np.sin(lat0) + np.cos(lat) * np.cos(lat0) * np.cos(lon - lon0)
    return EARTH_RADIUS_KM * np.arccos(cosine.clip(-1, 1))


sens = sens.assign_coords(
    distance_km=great_circle_km(sens["latitude"], sens["longitude"]),
    hidden_distance_km=great_circle_km(sens["hidden_latitude"], sens["hidden_longitude"]),
)
cell = sens.sel(scalar="cell")

## Radius of influence for the `cell` scalar

The first plot is the mean sensitivity in each 100 km distance bin, normalised by its
largest bin. The second is the cumulative fraction of the total sensitivity within a
given distance. A wider footprint shows up as a slower fall-off and as larger 50% and 90%
radii. `latent` is the total at the processor input, `processor` is what went through the processor (all of `latent` for `no_skip`), and
`input` is the model input, where anything beyond about 200 km (encoder cutoff plus decoder
neighbours) went through the processor.

In [ ]:
def radial_profile(values: xr.DataArray, distance: xr.DataArray, dim: str) -> xr.Dataset:
    binned = values.groupby_bins(distance, DISTANCE_BINS_KM, labels=DISTANCE_BINS_KM[1:])
    mean = binned.mean(dim)
    total = values.where(distance <= DISTANCE_BINS_KM[-1]).sum(dim)
    cumulative = binned.sum(dim).cumsum(f"{distance.name}_bins") / total
    return xr.Dataset({"mean": mean / mean.max(f"{distance.name}_bins"), "cumulative": cumulative})


def mark_radii_on_line_plot(ax, axis: str = "x") -> None:
    """A dark grey line at each graph radius, along ``axis``."""
    for (name, radius), style in zip(RADII_KM.items(), RADII_STYLES):
        line = ax.axvline if axis == "x" else ax.axhline
        line(radius, color="0.3", lw=0.8, ls=style, label=name)


def radius_containing(cumulative: xr.DataArray, fraction: float) -> xr.DataArray:
    dim = next(d for d in cumulative.dims if d.endswith("_bins"))
    return cumulative[dim].where(cumulative >= fraction).min(dim)


profiles = {
    "latent": radial_profile(cell["latent_sensitivity"], sens["hidden_distance_km"], "hidden"),
    "processor": radial_profile(cell["processor_sensitivity"], sens["hidden_distance_km"], "hidden"),
    "input": radial_profile(cell["input_total"], sens["distance_km"], "cell"),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, constrained_layout=True)
for column, (space, profile) in enumerate(profiles.items()):
    for label in CHECKPOINTS:
        p = profile.sel(checkpoint=label)
        p["mean"].plot(ax=axes[0, column], yscale="log", label=label)
        p["cumulative"].plot(ax=axes[1, column], label=label)
    axes[0, column].set(title=f"{space}: mean sensitivity / max bin", xlabel="")
    axes[1, column].set(title=f"{space}: cumulative fraction", xlabel="distance from target (km)")
    axes[1, column].axhline(0.5, color="gray", lw=0.5)
    axes[1, column].axhline(0.9, color="gray", lw=0.5)
for ax in axes.flat:
    mark_radii_on_line_plot(ax)
axes[0, 0].legend(fontsize=7)

pd.DataFrame(
    {
        f"{space} r{int(100 * f)} (km)": radius_containing(profile["cumulative"], f).to_pandas()
        for space, profile in profiles.items()
        for f in (0.5, 0.9)
    }
)

### Radius and share by input kind and input hour

Top left is the 90% radius of each kind's sensitivity at each input hour. The other three panels
are the share of input sensitivity by kind: top right sums every variable of a kind; bottom left
divides by the kind's number of input variables (22 per mission kind, 9 static, 3 time-varying
forcing); bottom right is taken at the encoder's input, each field's FieldEncoder latent
(8 channels) and the passthrough inputs (1 channel per variable), divided by channel count.
Count columns enter only through the observed mask (`counts > 0`), so their gradient is zero.
All gradients are with respect to normalised inputs, so the normalisations also enter the
comparison.

In [ ]:
by_kind = radial_profile(cell["input_sensitivity"], sens["distance_km"], "cell")
r90 = radius_containing(by_kind["cumulative"], 0.9)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
r90.stack(row=("checkpoint", "input_hour")).transpose("row", "kind").to_pandas().plot.bar(ax=axes[0, 0])
axes[0, 0].set(ylabel="r90 (km)", title="90% radius by kind")
mark_radii_on_line_plot(axes[0, 0], axis="y")
axes[0, 0].legend(fontsize=7, ncol=2)
plot_shares(axes.flat[1:], cell)

## Maps

The star is the target cell, and the dark grey circles mark the graph radii in `RADII_KM`: the
encoder's 100 km (dashed) and the processor's 1200 km (dotted). In the input maps, grey dots are
cells where sla was observed in that input bin.

In [ ]:
def draw(ax, values: xr.DataArray, **kwargs):
    """plots.draw, with the graph radii and the target on top."""
    image = plots.draw(ax, values, **kwargs)
    for radius, style in zip(RADII_KM.values(), RADII_STYLES):
        circle = Geodesic().circle(target_lon, target_lat, radius * 1e3, n_samples=360)
        ax.plot(*np.vstack([circle, circle[:1]]).T, color="0.2", lw=1, ls=style, transform=ccrs.Geodetic())
    ax.plot(target_lon, target_lat, "r*", ms=8, mec="w", mew=0.5, transform=ccrs.PlateCarree())
    return image


def nearby(xda: xr.DataArray, lat: str, lon: str) -> xr.DataArray:
    """Only the cells near the target, so the zoomed maps do not draw the whole globe."""
    window = (abs(xda[lat] - target_lat) < ZOOM_DEGREES + 2) & (
        abs((xda[lon] - target_lon + 180) % 360 - 180) < ZOOM_DEGREES + 2)
    return xda.where(window, drop=True)

### Processor input

In [ ]:
latent = normalised_log(cell["latent_sensitivity"], "hidden")

for extent in (None, ZOOM):
    fig, axes = map_axes(1, len(CHECKPOINTS), extent)
    for column, label in enumerate(CHECKPOINTS):
        values = latent.sel(checkpoint=label)
        image = draw(axes[0, column], values if extent is None else nearby(values, "hidden_latitude", "hidden_longitude"))
        axes[0, column].set_title(label)
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dz| / max)")

### Decoder input vs processor, `cell` scalar

Unlike the maps above, each panel keeps its absolute scale relative to the total latent
maximum, so the share of each path is visible. The decoder gradient should light up only
the target's 3 nearest hidden nodes. For `skip` it is the skip path; for `no_skip` it is at the
processor's output, and processor is the total.

In [ ]:
paths = {"latent_sensitivity": "total", "decoder_sensitivity": "decoder", "processor_sensitivity": "processor"}
reference = cell["latent_sensitivity"].max("hidden")

fig, axes = map_axes(len(CHECKPOINTS), len(paths), ZOOM)
for row, label in enumerate(CHECKPOINTS):
    for column, (name, title) in enumerate(paths.items()):
        values = np.log10(cell[name] / reference).clip(min=-4).sel(checkpoint=label)
        image = draw(axes[row, column], nearby(values, "hidden_latitude", "hidden_longitude"))
        axes[row, column].set_title(f"{label}: {title}")
fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|grad| / max total)")

(cell[list(paths)].sum("hidden") / cell["latent_sensitivity"].sum("hidden")).to_dataframe()[list(paths)]

### Model input, `cell` scalar

Sensitivity summed over all variables, at each input hour.

In [ ]:
total = normalised_log(cell["input_sensitivity"].sum("kind"), ["cell", "input_hour"])

fig, axes = map_axes(len(CHECKPOINTS), len(sens["input_hour"]), ZOOM)
for row, label in enumerate(CHECKPOINTS):
    for column, hour in enumerate(sens["input_hour"].values):
        image = draw(axes[row, column], nearby(total.sel(checkpoint=label, input_hour=hour), "latitude", "longitude"))
        plots.draw_seen(axes[row, column], nearby(sens["sla_seen"].sel(input_hour=hour), "latitude", "longitude"))
        axes[row, column].set_title(f"{label}: input hour {hour}")
fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")

### Model input by kind, `cell` scalar, t0 (input hour 0)

Hour 0 rather than the last input hour, which is the forcing-only step.
Count kinds are left out: their gradient is zero by construction.

In [ ]:
kinds = ["sla", "ugxt", "vgxt", "swh", "time_forcing", "static"]
last = cell["input_sensitivity"].sel(input_hour=0, kind=kinds)
last = normalised_log(last, "cell")

fig, axes = map_axes(len(CHECKPOINTS), len(kinds), ZOOM)
for row, label in enumerate(CHECKPOINTS):
    for column, kind in enumerate(kinds):
        image = draw(axes[row, column], nearby(last.sel(checkpoint=label, kind=kind), "latitude", "longitude"))
        axes[row, column].set_title(f"{label}: {kind}")
fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")